# Mineral AI: Hackathon Solution

## 1. Setup and Data Loading
First, I import nessesary modules.

In [1]:
import pandas as pd
import numpy as np
import mendeleev as ml
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler

C:\Users\Smart\Whiskas\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Second, I upload data from CSV-files.

In [3]:
train_df = pd.read_csv('train.csv')
X_train = train_df.drop(columns=['target']).fillna(0)
y_train = train_df['target']

## 2. Preprocessing
I change names of minerals in order to unify them and make them more readable.

In [4]:
y_new = y_train.copy()
for num, elem in enumerate(y_train):
    new_elem = elem.lower()
    if new_elem == '???':
        continue
    elif new_elem == 'other':
        new_elem = 'другое'
    elif new_elem == 'СУГМ':
        new_elem = 'слюдисто-углисто-глинистый минерал'
    elif new_elem == 'УРД':
        new_elem = 'углисто-растительный детрит'
    elif 'сфен' in new_elem:
        new_elem = 'титанит'
    elif '(' in new_elem:
        if '?' in new_elem:
            new_elem = new_elem[:new_elem.index('(')].strip(' ')
        else:
            new_elem = new_elem[new_elem.index('(')+1:new_elem.index(')')]
    elif 'мат' in new_elem or 'мин' in new_elem:
        if 'кар' in new_elem and 'кр' in new_elem and 'гл' in new_elem:
            new_elem = 'карбонатно-кремнисто-глинистый минерал'
        elif 'кар' in new_elem and 'кр' in new_elem:
            new_elem = 'карбонатно-кремнистый минерал'
        elif 'кр' in new_elem and 'гл' in new_elem:
            new_elem = 'кремнисто-глинистый минерал'
        elif 'кар' in new_elem and 'гл' in new_elem:
            new_elem = 'карбонатно-глинистый минерал'
        elif 'кар' in new_elem:
            new_elem = 'карбонатный минерал'
        elif 'гл' in new_elem:
            new_elem = 'глинистый минерал'
        elif 'кр' in new_elem:
            new_elem = 'кремнистый минерал'
        elif 'кр' in new_elem and 'ф' in new_elem:
            new_elem = 'кремнисто-фосфатный минерал'
        elif 'жел' in new_elem:
            new_elem = 'железистый минерал'
        elif 'тит' in new_elem:
            new_elem = 'титанистый минерал'
        elif 'фос' in new_elem:
            new_elem = 'фосфатный минерал'
        elif 'уг' in new_elem:
            new_elem = 'углистый минерал'
        elif 'хл' in new_elem:
            new_elem = 'хлоросодержащий минерал'
        elif 'слюд' in new_elem:
            new_elem = 'слюд. минерал'
    elif 'це' in new_elem or 'ц.' in new_elem:
        if '+' in new_elem:
            if 'к' in new_elem:
                new_elem = 'кварц + глинистый цемент'
            if 'п' in new_elem:
                new_elem = 'полевой шпат + глинистый цемент'
        else:
            if 'гл' in new_elem and 'кр' in new_elem:
                new_elem = 'кремнисто-глинистый цемент'
            elif 'гл' in new_elem and 'кар' in new_elem:
                new_elem = 'карбонатно-глинистый цемент'
            elif 'гл' in new_elem:
                new_elem = 'глинистый цемент'
            elif 'кр' in new_elem:
                new_elem = 'кремнистый цемент'
            elif 'кар' in new_elem:
                new_elem = 'карбонатный цемент'
    elif 'гид' in new_elem and 'сл' in new_elem:
        if 'сидер' in new_elem:
            new_elem = 'сидеритизированная гидрослюда'
        else:
            new_elem = 'гидрослюда'
    elif 'поле' in new_elem:
        if 'к' in new_elem:
            new_elem = 'калиевый полевой шпат'
        else:
            new_elem = 'полевой шпат'
    elif 'муск' in new_elem:
        new_elem = 'мусковит'
    elif 'ожел' in new_elem:
        new_elem = 'ожелезненный' + new_elem[new_elem.index(' '):]
    y_new[num] = new_elem
y_train = y_new
print('Preprocessing of y complete.')

Preprocessing of y complete.


Then I add lists of the criteria I'm going to use to tell different minerals apart.

In [5]:
PERCENTAGES = (
('S', 'Ca', 'O'),
('Na', 'Cl'),
('Fe', 'Al'),
('Si', 'O'),
('C', 'O'),
('Si', 'Al', 'K', 'Ca', 'Fe'),
('Fe', 'Mg', 'Mn'),
('Ca', 'F'),
('Ca', 'P', 'F', 'O'),
('Si', 'Al', 'Mg', 'Fe', 'O'),
('Ca', 'C', 'O'),
('Ca', 'Mg', 'C', 'O'),
('Fe', 'C', 'O'),
('Mg', 'Fe', 'K', 'Na', 'Ca'),
('Si', 'Al', 'O'),
('Si', 'P', 'O'),
('Si', 'Al', 'Ca', 'Mg', 'C', 'O'),
('Si', 'Ca', 'Mg', 'C', 'O'),
('Fe' ,'Ca', 'Mg', 'C', 'O'),
('Ca', 'Fe', 'C', 'O'),
('Fe', 'O'),
('K', 'Na', 'Ca'),
('Si', 'Al', 'K', 'O'),
('Si', 'Al', 'K', 'C', 'O'),
('Si', 'Al', 'K', 'Mg', 'Fe', 'O')
)

RELATIONS = (
('Fe', ('Fe', 'Al')),
('Si', 'Al'),
('Na', 'Al'),
('Na', ('Na', 'Ca', 'K')),
('S', 'Ca'),
('S', 'O'),
('Ca', 'O'),
('Ca', ('Fe', 'Mg', 'Mn')),
('Ca', 'P'),
('Ba','S'),
('K', 'Al'),
(('Fe', 'Mg'), 'K'),
('Na', 'Cl'),
('Al', ('K', 'Na')),
(('Mg', 'Fe'), 'Al'),
('S', 'Ca'),
('S', 'O'),
('Fe', 'Al'),
('K', ('Fe', 'Mg')),
('Ca', 'Mg'),
('Ca', 'C'),
('Si', 'O'),
('Mg', 'Fe'),
('Al', 'Si'),
('Ti', 'Fe'),
('Fe', 'Al'),
('Fe', 'S'),
('Ti', 'O'),
('Fe', 'C'),
('Ca', 'S'),
(('Zn', 'Fe'), 'S'),
('Ca', 'Ti'),
('Si', 'Ti'),
(('Ti', 'Nb'), 'Ti'),
('C', 'S'),
('O', 'C'),
(('Ca', 'Mg'), 'C'),
('Mg', 'C'),
('Ca', 'F'),
('Ca', ('F', 'O')),
(('Ce', 'La'), 'P'),
('Fe', 'P'),
('Al', 'P'),
('P', 'F'),
('K', 'Cl'),
('Ca', 'Cl'),
('P', 'Cl'),
(('Na', 'K'), 'Cl'),
('Si', 'Cl'),
('Fe', 'O'),
('Fe', 'Si'),
(('Fe', 'Al'), 'Si'),
('Mn', 'C'),
('Zn', 'C'),
('Si', ('Si', 'O')),
('Si', 'P'),
(('Ca', 'Mg'), ('Si', 'Al')),
(('Ca', 'Mg'), 'Si'),
('O', ('Si', 'C')),
('Si', ('K', 'Na', 'Ca')),
(('K', 'Na', 'Ca', 'Mg', 'Fe'), 'Al'),
('Ca', ('Mg', 'Fe')),
(('Ca', 'Mg', 'Fe'), 'C'),
('Fe', ('Mg', 'Fe')),
('Ca', ('Fe', 'Mg')),
('Ca', 'Fe'),
(('Ca', 'Fe'), 'C'),
(('K', 'Na', 'Ca'), 'Al'),
('K', ('K', 'Na')),
('Ca', ('Ca', 'Na')),
('K', ('K', 'Na', 'Ca')),
('C', ('Si', 'Al')),
(('Si', 'Al'), 'O')
)

After that I change my data table: fill all NaN values with 0, add new criteria listed before and standardize all values.

In [6]:
def magic(init_df):
    df = init_df.copy()
    df = df.fillna(0)
    new_df = dict()

    while np.any(df > 100):
        df[df > 100] /= 10

    for el in ['O', 'S']:
        df[f'{el}'] += df[f'{el}_alt']
        df = df.drop(columns=[f'{el}_alt'])

    new_df['total'] = df.sum(axis=1)

    for col in df.columns:
        new_df[f'{col}'] = df[col]
        df[f'{col}_at'] = df[col] / ml.element(col).atomic_weight

    for n, d in RELATIONS:
        if isinstance(n, tuple):
            n_str = '(' + ' + '.join(n) + ')'
            n_sum = None
            for el in n:
                if n_sum is not None:
                    n_sum += df[f'{el}_at']
                else:
                    n_sum = df[f'{el}_at']
        else:
            n_sum = df[f'{n}_at']
            n_str = n
        if isinstance(d, tuple):
            d_str = '(' + ' + '.join(d) + ')'
            d_sum = None
            for el in d:
                if d_sum is not None:
                    d_sum += df[f'{el}_at']
                else:
                    d_sum = df[f'{el}_at']
        else:
            d_sum = df[f'{d}_at']
            d_str = d

        rels = n_sum / d_sum
        rels[rels == np.inf] = 0
        rels.fillna(0)
        new_df[f'{n_str} / {d_str}'] = rels

    for percentage in PERCENTAGES:
        percentage_str = '(' + ' + '.join(percentage) + ')'
        percentage_sum = None
        for el in percentage:
            if percentage_sum is not None:
                percentage_sum += df[f'{el}']
            else:
                percentage_sum = df[f'{el}']
        new_df[f'{percentage_str}'] = percentage_sum

    ss = StandardScaler()

    return ss.fit_transform(pd.DataFrame(new_df).fillna(0))

X_train = magic(X_train)
print('Preprocessing of x complete as well.')

Preprocessing of x complete as well.


## 3. Model Training
I use a Random Forest classifier. To find the best parameters possible, I use GridSearchCV. I divide all the data into only 2 parts because my least populated class consists of only 2 members, so StratifiedKFold cannot maintain equivalency between all classes' shares when n_split > 2.

In [7]:
param_grid = {
'n_estimators': [100, 300, 500, 800],
'max_depth': [None, 10, 20, 30],
'min_samples_split': [2, 5, 10],
'min_samples_leaf': [1, 2, 4],
}

rf = RandomForestClassifier(random_state=42)
skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
search = GridSearchCV(
estimator=rf,
param_grid=param_grid,
cv=skf,
scoring='f1_macro',
n_jobs=-1,
return_train_score=True
)

search.fit(X_train, y_train)

best_params = search.best_params_
print(f"Промежуточные параметры: {best_params}")
print(f"Промежуточный результат: {search.best_score_:.4f}")

KeyboardInterrupt: 

After that I use GridSearchCV once more to find even better parameters based on what I've already found.

In [8]:
nest = best_params['n_estimators']
deep = best_params['max_depth']
if deep is None:
    param_grid = {
        'n_estimators': [int(val) for val in range(int(0.8 * nest), int(1.21 * nest), int(0.05 * nest))]
    }

    rf = RandomForestClassifier(
        max_depth=None,
        min_samples_split=best_params['min_samples_split'],
        min_samples_leaf=best_params['min_samples_leaf'],
        random_state=42
    )
else:
    param_grid = {
    'n_estimators': [int(val) for val in range(int(0.8 * nest), int(1.21 * nest), int(0.05 * nest))],
    'max_depth': [int(val) for val in range(int(0.8 * deep), int(1.21 * deep), int(0.1 * deep))],
    }

    rf = RandomForestClassifier(
    min_samples_split=best_params['min_samples_split'],
    min_samples_leaf=best_params['min_samples_leaf'],
    random_state=42
    )
skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
search = GridSearchCV(
estimator=rf,
param_grid=param_grid,
cv=skf,
scoring='f1_macro',
n_jobs=-1,
return_train_score=True
)

search.fit(X_train, y_train)

final_best_params = search.best_params_
final_best_params['min_samples_split'] = best_params['min_samples_split']
final_best_params['min_samples_leaf'] = best_params['min_samples_leaf']
if deep is None: final_best_params['max_depth'] = None
print(f"Лучшие параметры: {final_best_params}")
print(f"Лучший результат: {search.best_score_:.4f}")

Лучшие параметры: {'n_estimators': 270, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': None}
Лучший результат: 0.5441


In [10]:
best_rf = RandomForestClassifier(**{'n_estimators': 270, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': None}, random_state=42)
best_rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",270
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

Finally, it's time for me to train my model.

In [9]:
best_rf = RandomForestClassifier(**final_best_params, random_state=42)
best_rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",270
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

## 4. Generating Predictions
After all this I make my predictions and save them into a CSV-file.

In [10]:
test_df = magic(pd.read_csv('test_features.csv'))
preds = pd.DataFrame()
preds['id'] = list(range(1, 4001))
preds['Mineral_name'] = best_rf.predict(test_df)
preds.to_csv('predictions.csv', index=False, header=['id', 'Mineral_Name'])
print('Submission saved as predictions.csv')

Submission saved as predictions.csv
